# Day 5: 動的利用者均衡 (DUE) vs 動的システム最適 (DSO)

## 学習目標
- 「自己中心的な経路選択の均衡」(DUE) と「ネットワーク全体の総旅行時間を最小化する配分」(DSO) の違いを理解する
- `uxsim.DTAsolvers.SolverDUE` / `SolverDSO_D2D` を使って同じネットワークの2つの解を求める
- 両者の総旅行時間 (Total Travel Time, TTT) を比較し、Price of Anarchy（自己中心性のコスト）を数値で確認する

## 前提知識
- Day 3 で扱った DUO（逐次的な近似解法）の発展版。DUE/DSOソルバーは
  「日々の経路選択の繰り返し（day-to-day dynamics）」をシミュレーションし、
  収束した定常状態を均衡解の近似として得る、という考え方です。
- DUE: 各車両が「自分の旅行時間」を最小化しようとする → Day 3 のDUOが目指す先の理論的な状態
- DSO: 各車両が「自分の旅行時間 + 自分が他人に与える外部性（混雑コスト）」を最小化しようとする
  → 社会的に最も効率的な配分


In [ ]:
from uxsim import World
from uxsim.DTAsolvers import SolverDUE, SolverDSO_D2D

def func_World():
    """ボトルネックのある2経路ネットワーク。SolverはこのWorldを繰り返し生成して day-to-day dynamics を回す"""
    W = World(
        name="",
        deltan=5,
        tmax=4000,
        print_mode=0, save_mode=0, show_mode=0,
        random_seed=None,
    )
    W.addNode("orig", 0, 0)
    W.addNode("mid1", 1, 1)
    W.addNode("mid2", 1, -1)
    W.addNode("dest", 2, 0)

    # 経路1: 短いが1車線(詰まりやすい)
    W.addLink("o_mid1", "orig", "mid1", length=1000, free_flow_speed=20, number_of_lanes=1)
    W.addLink("mid1_d", "mid1", "dest", length=1000, free_flow_speed=20, number_of_lanes=1)
    # 経路2: 遠回りだが2車線(余裕あり)
    W.addLink("o_mid2", "orig", "mid2", length=2000, free_flow_speed=20, number_of_lanes=2)
    W.addLink("mid2_d", "mid2", "dest", length=2000, free_flow_speed=20, number_of_lanes=2)

    W.adddemand("orig", "dest", t_start=0, t_end=3000, flow=0.8)
    return W

solver_due = SolverDUE(func_World)
W_due = solver_due.solve(max_iter=30, n_routes_per_od=5, swap_prob=0.05)
print("=== DUE (自己中心的均衡) ===")
W_due.analyzer.print_simple_stats(force_print=True)


In [ ]:
solver_dso = SolverDSO_D2D(func_World)
W_dso = solver_dso.solve(max_iter=30, n_routes_per_od=5, swap_prob=0.05)
print("=== DSO (システム最適配分) ===")
W_dso.analyzer.print_simple_stats(force_print=True)


In [ ]:
ttt_due = solver_due.W_sol.analyzer.total_travel_time
ttt_dso = solver_dso.W_sol.analyzer.total_travel_time
poa = ttt_due / ttt_dso

print(f"DUE 総旅行時間 (TTT): {ttt_due:.1f} s")
print(f"DSO 総旅行時間 (TTT): {ttt_dso:.1f} s")
print(f"Price of Anarchy (DUE/DSO): {poa:.3f}")
print("→ 1.0に近いほど、自己中心的な経路選択でも社会的に効率的な状態に近いことを意味する")


## Part B: 演習

1. `swap_prob` や `max_iter` を変えて、DUE/DSOそれぞれの収束の様子（`solver.ttts` に
   各イテレーションの総旅行時間が記録されています）をプロットしてください。
2. 経路2（`o_mid2`/`mid2_d`）の車線数を1車線に減らし、2経路がほぼ対称なケースで
   Price of Anarchyがどう変わるか実験してください。ボトルネックの非対称性が
   大きいほどPoAが大きくなるか、小さくなるか予想してから確認しましょう。


In [ ]:
import matplotlib.pyplot as plt

plt.plot(solver_due.ttts, label="DUE")
plt.plot(solver_dso.ttts, label="DSO")
plt.xlabel("iteration (day)")
plt.ylabel("total travel time [s]")
plt.legend()
plt.grid()
plt.show()

# TODO: 経路2を1車線にしたときのPoAを求めてみる


## Part C: 考察

- Day 3 の最後で立てた問い「DUOはDSOと同じ結果になるか」に、自分の言葉で答えてください。
- `merge_priority`（Day 1）・信号のオフセット（Day 2）・congestion_pricing（Day 7で扱う課金）は、
  それぞれ「DUEをDSOに近づけるための現実の政策手段」と見なせます。それぞれがどう効くか
  一言でまとめてください。
